In [2]:
!pip install -U langchain langchain-openai langgraph

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.5/161.5 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.1/125.1 kB 6.7 MB/s eta 0:00:00
  Attempting uninstall: langchain
    Found existing installation: langchain 1.3.17
    Uninstalling langchain-1.3.17:
      Successfully uninstalled langchain-1.3.17


In [3]:
from google.colab import userdata

OPENROUTER_API_KEY = userdata.get("Exp_trackker")

print("API key loaded:", bool(OPENROUTER_API_KEY))

if OPENROUTER_API_KEY:
    print("Key prefix:", OPENROUTER_API_KEY[:10])
    print("Key length:", len(OPENROUTER_API_KEY))

API key loaded: True
Key prefix: sk-or-v1-a
Key length: 73


In [4]:
import requests

response = requests.post(
    "https://openrouter.ai/api/v1/chat/completions",
    headers={
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json"
    },
    json={
        "model": "openai/gpt-oss-20b",
        "messages": [
            {
                "role": "user",
                "content": "Say only: OPENROUTER TEST SUCCESS"
            }
        ],
        "max_tokens": 20
    }
)

print("Status:", response.status_code)
print("Response:", response.text[:1000])

Status: 200
Response: {"id":"gen-1788865211-7ZSvIek8zlh4oHOzzh7c","object":"chat.completion","created":1788865211,"model":"openai/gpt-oss-20b","provider":"CoreWeave","system_fingerprint":null,"service_tier":null,"choices":[{"index":0,"logprobs":null,"finish_reason":"length","native_finish_reason":"length","message":{"role":"assistant","content":null,"refusal":null,"reasoning":"The user has asked: \"Say only: OPENROUTER TEST SUCCESS\". We","reasoning_details":[{"type":"reasoning.text","text":"The user has asked: \"Say only: OPENROUTER TEST SUCCESS\". We","format":"unknown","index":0}]}}],"usage":{"prompt_tokens":74,"completion_tokens":20,"total_tokens":94,"cost":0.00000482,"is_byok":false,"prompt_tokens_details":{"cached_tokens":32,"cache_write_tokens":0,"audio_tokens":0,"video_tokens":0},"cost_details":{"upstream_inference_cost":0.00000482,"upstream_inference_prompt_cost":0.00000222,"upstream_inference_completions_cost":0.0000026},"completion_tokens_details":{"reasoning_tokens":15,"ima

In [5]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="openai/gpt-oss-20b",
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1",
    temperature=0
)

print("✅ LangChain LLM created successfully")

✅ LangChain LLM created successfully


In [6]:
response = llm.invoke(
    "Reply with exactly: LANGCHAIN TEST SUCCESS"
)

print(response.content)

LANGCHAIN TEST SUCCESS


In [7]:
import csv
import os

CSV_FILE = "expenses.csv"

def initialize_csv():
    if not os.path.exists(CSV_FILE):
        with open(CSV_FILE, "w", newline="") as file:
            writer = csv.writer(file)
            writer.writerow(["Item", "Amount", "Category"])

initialize_csv()

print("✅ Expense CSV initialized successfully")

✅ Expense CSV initialized successfully


In [8]:
def add_expense(item, amount, category):
    with open(CSV_FILE, "a", newline="") as file:
        writer = csv.writer(file)
        writer.writerow([item, amount, category])

    return f"Expense added: {item} - ₹{amount} ({category})"


def get_expenses():
    expenses = []

    with open(CSV_FILE, "r", newline="") as file:
        reader = csv.DictReader(file)

        for row in reader:
            expenses.append(row)

    return expenses


print("✅ Expense functions created successfully")

✅ Expense functions created successfully


In [9]:
from langchain_core.tools import StructuredTool

add_expense_tool = StructuredTool.from_function(
    func=add_expense,
    name="add_expense",
    description="Add an expense with item name, amount, and category."
)

get_expenses_tool = StructuredTool.from_function(
    func=get_expenses,
    name="get_expenses",
    description="Get all recorded expenses from the expense CSV file."
)

tools = [
    add_expense_tool,
    get_expenses_tool
]

print("✅ Expense tools created successfully")

✅ Expense tools created successfully


In [10]:
from langgraph.prebuilt import create_react_agent

agent = create_react_agent(
    model=llm,
    tools=tools,
    prompt="""
You are an expense tracking assistant.

You can:
1. Add expenses using the add_expense tool.
2. Retrieve expenses using the get_expenses tool.

When the user asks you to add an expense, use the add_expense tool.
When the user asks to see expenses, use the get_expenses tool.

Always be clear and concise.
"""
)

print("✅ LangGraph expense agent created successfully")

✅ LangGraph expense agent created successfully


/tmp/ipykernel_599/1128343261.py:3: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


In [11]:
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Add an expense for Pizza costing 300 rupees under Food."
        }
    ]
})

print(result["messages"][-1].content)

✅ Expense added: **Pizza** – ₹300 (Food).


In [12]:
print("📋 Current Expenses:")
print(get_expenses())

📋 Current Expenses:
[{'Item': 'Pizza', 'Amount': '300', 'Category': 'Food'}]
